# ResNet-18 Phase 2 v2: Joint Training with Proper Validation

## Fixes from v1:
- **Train/Val Split**: 80% train, 20% validation
- **Data Augmentation**: RandomResizedCrop, HorizontalFlip, ColorJitter
- **Validation Tracking**: Monitor generalization during training
- **Early Stopping**: Stop when validation accuracy plateaus
- **Mask Regularization**: Prevent mask collapse to near-zero

## Research Question
What frequency preferences are built into the ResNet-18 **architecture** (not learned from data)?

---

## 1. Setup and Imports

In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/bab61wot/VIT/SIM2REAL


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import time

from frequency.transforms import apply_fft, apply_ifft
from frequency.mask import Learnable2DFrequencyMask

print("Imports successful")

Imports successful


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce GTX 1060 6GB
Memory: 6.36 GB


## 2. Load Dataset with Train/Val Split and Augmentation

In [6]:
from datasets import load_from_disk
from torchvision import transforms
from data.dataset import HFImageNetDataset

CACHE_PATH = os.path.join(PROJECT_ROOT, "data", "imagenet_25k_cache")
BATCH_SIZE = 64
TRAIN_SPLIT = 0.8

print(f"Loading dataset from {CACHE_PATH}...")
imagenet_subset = load_from_disk(CACHE_PATH)
print(f"Loaded {len(imagenet_subset):,} images")

# Training transforms WITH augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transforms WITHOUT augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create full dataset (we'll apply transforms per-split)
full_dataset = HFImageNetDataset(imagenet_subset, transform=None)

# Split into train and validation
train_size = int(TRAIN_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size

# Use generator for reproducibility
generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(range(len(full_dataset)), [train_size, val_size], generator=generator)

print(f"Train: {len(train_indices):,} | Val: {len(val_indices):,}")

Loading dataset from /home/bab61wot/VIT/SIM2REAL/data/imagenet_25k_cache...
Loaded 25,000 images
Train: 20,000 | Val: 5,000


In [5]:
# Custom dataset wrapper to apply different transforms
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, indices, transform):
        self.hf_dataset = hf_dataset
        self.indices = list(indices)
        self.transform = transform
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        item = self.hf_dataset[real_idx]
        image = item['image']
        label = item['label']
        
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

train_dataset = TransformSubset(imagenet_subset, train_indices.indices, train_transform)
val_dataset = TransformSubset(imagenet_subset, val_indices.indices, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

Train loader: 313 batches
Val loader: 79 batches


## 3. Create Joint Training Pipeline with Mask Regularization

In [7]:
class JointTrainingPipeline(nn.Module):
    """
    Phase 2 Pipeline with mask regularization to prevent collapse.
    """
    
    def __init__(self, classifier, image_size=224):
        super().__init__()
        
        self.classifier = classifier
        for param in self.classifier.parameters():
            param.requires_grad = True
        
        self.freq_mask = Learnable2DFrequencyMask(
            image_size=image_size,
            init_value=1.0,
            init_std=0.1
        )
        
        mask_params = sum(p.numel() for p in self.freq_mask.parameters())
        classifier_params = sum(p.numel() for p in self.classifier.parameters())
        
        print(f"\nJointTrainingPipeline created:")
        print(f"  Frequency mask: {mask_params:,} trainable params")
        print(f"  Classifier: {classifier_params:,} trainable params")
    
    def forward(self, images):
        fft_result = apply_fft(images)
        masked_fft = self.freq_mask(fft_result)
        reconstructed = apply_ifft(masked_fft)
        outputs = self.classifier(reconstructed)
        return outputs, reconstructed
    
    def get_mask_params(self):
        return self.freq_mask.parameters()
    
    def get_classifier_params(self):
        return self.classifier.parameters()
    
    def get_mask_visualization(self):
        return self.freq_mask.get_mask_visualization()
    
    def mask_regularization_loss(self):
        """Regularization to keep mask values near 1.0 (prevent collapse)"""
        mask = self.freq_mask.mask_weights  # Fixed: use mask_weights not mask
        # Penalize deviation from 1.0
        return torch.mean((mask - 1.0) ** 2)

In [8]:
print("Loading ResNet-18 with random initialization...")
classifier = models.resnet18(weights=None, num_classes=1000)
print(f"  Parameters: {sum(p.numel() for p in classifier.parameters()):,}")

pipeline = JointTrainingPipeline(classifier, image_size=224)
pipeline = pipeline.to(device)
print(f"\nPipeline on {device}")

Loading ResNet-18 with random initialization...
  Parameters: 11,689,512

JointTrainingPipeline created:
  Frequency mask: 50,176 trainable params
  Classifier: 11,689,512 trainable params

Pipeline on cuda


## 4. Training Configuration

In [9]:
# Hyperparameters
EPOCHS = 90
MASK_LR = 0.01
CLASSIFIER_LR = 0.001
WEIGHT_DECAY = 0.0001
LR_STEP = 30
LR_GAMMA = 0.1
MASK_REG_WEIGHT = 0.01  # Regularization to prevent mask collapse
EARLY_STOP_PATIENCE = 15  # Stop if no improvement for 15 epochs

optimizer = optim.Adam([
    {'params': pipeline.get_mask_params(), 'lr': MASK_LR},
    {'params': pipeline.get_classifier_params(), 'lr': CLASSIFIER_LR}
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)
criterion = nn.CrossEntropyLoss()

print("Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Mask LR: {MASK_LR}")
print(f"  Classifier LR: {CLASSIFIER_LR}")
print(f"  Mask regularization: {MASK_REG_WEIGHT}")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")

Training Configuration:
  Epochs: 90
  Batch size: 64
  Mask LR: 0.01
  Classifier LR: 0.001
  Mask regularization: 0.01
  Early stopping patience: 15


In [10]:
RESULTS_DIR = Path(PROJECT_ROOT) / "experiments" / "results" / "resnet18_phase2_v2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results: {RESULTS_DIR}")

Results: /home/bab61wot/VIT/SIM2REAL/experiments/results/resnet18_phase2_v2


## 5. Training Loop with Validation

In [11]:
def evaluate(pipeline, dataloader, criterion, device):
    """Evaluate on validation set"""
    pipeline.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs, _ = pipeline(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), 100.0 * correct / total

In [12]:
# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}
best_val_acc = 0.0
patience_counter = 0
start_epoch = 0

# Check for checkpoint
checkpoint_path = RESULTS_DIR / "checkpoint.pt"
if checkpoint_path.exists():
    print("Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    pipeline.load_state_dict(checkpoint['pipeline_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    best_val_acc = checkpoint['best_val_acc']
    patience_counter = checkpoint.get('patience_counter', 0)
    history = checkpoint['history']
    print(f"Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.2f}%")
else:
    print("Starting fresh training")

print(f"\nTraining for {EPOCHS} epochs (from {start_epoch})...")
print("=" * 70)

Loading checkpoint...
Resumed from epoch 30, best_val_acc=7.72%

Training for 90 epochs (from 30)...


In [13]:
# Main training loop
for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    pipeline.train()
    
    train_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs, _ = pipeline(images)
        
        # Classification loss + mask regularization
        cls_loss = criterion(outputs, labels)
        reg_loss = pipeline.mask_regularization_loss()
        loss = cls_loss + MASK_REG_WEIGHT * reg_loss
        
        loss.backward()
        optimizer.step()
        
        train_loss += cls_loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx % 20 == 0:
            pbar.set_postfix({'loss': f"{cls_loss.item():.3f}", 'acc': f"{100.*correct/total:.1f}%"})
    
    scheduler.step()
    
    # Validation
    val_loss, val_acc = evaluate(pipeline, val_loader, criterion, device)
    
    # Epoch stats
    train_loss /= len(train_loader)
    train_acc = 100.0 * correct / total
    epoch_time = time.time() - epoch_start
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Get mask stats
    mask_viz = pipeline.get_mask_visualization()
    
    print(f"\nEpoch {epoch+1}/{EPOCHS} ({epoch_time:.1f}s)")
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
    print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
    print(f"  Mask:  mean={mask_viz.mean():.3f}, std={mask_viz.std():.3f}")
    print(f"  Gap:   {train_acc - val_acc:.2f}% (train - val)")
    
    # Check for improvement
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        patience_counter = 0
        print(f"  ** New best validation accuracy: {best_val_acc:.2f}% **")
        torch.save({'pipeline_state_dict': pipeline.state_dict(), 'val_acc': val_acc},
                   RESULTS_DIR / "best_model.pt")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")
    
    # Save checkpoint
    torch.save({
        'epoch': epoch + 1,
        'pipeline_state_dict': pipeline.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'patience_counter': patience_counter,
        'history': history
    }, checkpoint_path)
    
    # Visualize mask every 10 epochs
    if (epoch + 1) % 10 == 0:
        plt.figure(figsize=(8, 6))
        plt.imshow(mask_viz, cmap='RdBu_r', vmin=0.5, vmax=1.5)
        plt.colorbar(label='Mask Weight')
        plt.title(f'Mask - Epoch {epoch+1} (Val Acc: {val_acc:.2f}%)')
        plt.savefig(RESULTS_DIR / f"mask_epoch_{epoch+1}.png", dpi=150, bbox_inches='tight')
        plt.show()
    
    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}!")
        break
    
    print("-" * 70)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print("=" * 70)

Epoch 31/90: 100%|█████████████████████████████████| 313/313 [04:37<00:00,  1.13it/s, loss=1.431, acc=69.3%]



Epoch 31/90 (314.5s)
  Train: Loss=1.4312, Acc=69.45%
  Val:   Loss=6.3440, Acc=8.50%
  Mask:  mean=0.010, std=0.099
  Gap:   60.95% (train - val)
  ** New best validation accuracy: 8.50% **
----------------------------------------------------------------------


Epoch 32/90: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 313/313 [03:05<00:00,  1.68it/s, loss=1.314, acc=75.9%]



Epoch 32/90 (208.0s)
  Train: Loss=1.2265, Acc=75.92%
  Val:   Loss=6.3571, Acc=8.56%
  Mask:  mean=0.010, std=0.095
  Gap:   67.36% (train - val)
  ** New best validation accuracy: 8.56% **
----------------------------------------------------------------------


Epoch 33/90: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 313/313 [03:05<00:00,  1.69it/s, loss=1.081, acc=77.7%]



Epoch 33/90 (208.6s)
  Train: Loss=1.1551, Acc=77.67%
  Val:   Loss=6.3856, Acc=9.08%
  Mask:  mean=0.010, std=0.093
  Gap:   68.59% (train - val)
  ** New best validation accuracy: 9.08% **
----------------------------------------------------------------------


Epoch 34/90: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 313/313 [03:05<00:00,  1.69it/s, loss=1.115, acc=79.6%]



Epoch 34/90 (207.5s)
  Train: Loss=1.0933, Acc=79.56%
  Val:   Loss=6.4153, Acc=8.54%
  Mask:  mean=0.010, std=0.091
  Gap:   71.02% (train - val)
  No improvement (1/15)
----------------------------------------------------------------------


Epoch 35/90:  58%|████████████████████████████████████████████████████████████████████████████████████████████▌                                                                    | 180/313 [01:50<01:21,  1.64it/s, loss=1.416, acc=80.7%]


KeyboardInterrupt: 

## 6. Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], label='Train', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('ResNet-18 Phase 2 v2: Train vs Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_history.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Stats:")
print(f"  Train Acc: {history['train_acc'][-1]:.2f}%")
print(f"  Val Acc:   {history['val_acc'][-1]:.2f}%")
print(f"  Best Val:  {best_val_acc:.2f}%")

## 7. Load Best Model and Visualize Mask

In [ ]:
# Load best model
best_checkpoint = torch.load(RESULTS_DIR / "best_model.pt", map_location=device)
pipeline.load_state_dict(best_checkpoint['pipeline_state_dict'])
print(f"Loaded best model (Val Acc: {best_checkpoint['val_acc']:.2f}%)")

learned_mask = pipeline.get_mask_visualization()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(learned_mask, cmap='RdBu_r', vmin=0.5, vmax=1.5)
ax.set_title(f'Phase 2 v2 Learned Mask\n(Val Acc: {best_checkpoint["val_acc"]:.2f}%)', fontweight='bold', fontsize=14)
ax.axis('off')
plt.colorbar(im, ax=ax, label='Mask Weight')
plt.savefig(RESULTS_DIR / "learned_mask.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMask Statistics:")
print(f"  Min:  {learned_mask.min():.4f}")
print(f"  Max:  {learned_mask.max():.4f}")
print(f"  Mean: {learned_mask.mean():.4f}")
print(f"  Std:  {learned_mask.std():.4f}")

## 8. Compare with Phase 1

In [ ]:
phase1_dir = Path(PROJECT_ROOT) / "experiments" / "results" / "resnet18"
phase1_mask_path = phase1_dir / "learned_mask.pt"

if phase1_mask_path.exists():
    # Load Phase 1 mask
    phase1_mask_module = Learnable2DFrequencyMask(image_size=224)
    phase1_mask_module.load_state_dict(torch.load(phase1_mask_path, map_location='cpu'))
    phase1_mask = phase1_mask_module.get_mask_visualization()
    phase2_mask = learned_mask
    
    # Radial profile function
    def radial_profile(mask):
        h, w = mask.shape
        cy, cx = h // 2, w // 2
        y, x = np.ogrid[:h, :w]
        r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
        r_max = int(np.sqrt(cx ** 2 + cy ** 2))
        profile = []
        for rad in range(r_max):
            ring = (r >= rad) & (r < rad + 1)
            if ring.sum() > 0:
                profile.append(mask[ring].mean())
        return profile
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    im0 = axes[0].imshow(phase1_mask, cmap='RdBu_r', vmin=0.5, vmax=1.5)
    axes[0].set_title('Phase 1 (Pretrained, frozen)', fontweight='bold')
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(phase2_mask, cmap='RdBu_r', vmin=0.5, vmax=1.5)
    axes[1].set_title('Phase 2 v2 (From scratch)', fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1])
    
    diff = phase2_mask - phase1_mask
    vmax = max(abs(diff.min()), abs(diff.max()))
    im2 = axes[2].imshow(diff, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axes[2].set_title('Difference (P2 - P1)', fontweight='bold')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2])
    
    axes[3].plot(radial_profile(phase1_mask), label='Phase 1', linewidth=2)
    axes[3].plot(radial_profile(phase2_mask), label='Phase 2 v2', linewidth=2)
    axes[3].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    axes[3].set_xlabel('Frequency (0=DC, higher=high freq)')
    axes[3].set_ylabel('Mask Weight')
    axes[3].set_title('Radial Profiles', fontweight='bold')
    axes[3].legend()
    axes[3].grid(True, alpha=0.3)
    
    plt.suptitle('ResNet-18: Phase 1 vs Phase 2 v2', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "phase1_vs_phase2.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Correlation
    correlation = np.corrcoef(phase1_mask.flatten(), phase2_mask.flatten())[0, 1]
    print(f"\n=== Phase 1 vs Phase 2 v2 Analysis ===")
    print(f"Mask correlation: {correlation:.4f}")
    print()
    if correlation > 0.7:
        print("HIGH correlation -> Frequency preference is ARCHITECTURAL")
    elif correlation > 0.4:
        print("MODERATE correlation -> Mix of architectural and learned")
    else:
        print("LOW correlation -> Frequency preference is LEARNED from data")
else:
    print("Phase 1 results not found")

## 9. Save Results

In [ ]:
torch.save(pipeline.freq_mask.state_dict(), RESULTS_DIR / "learned_mask.pt")
torch.save(history, RESULTS_DIR / "training_history.pt")

with open(RESULTS_DIR / "summary.txt", 'w') as f:
    f.write("ResNet-18 Phase 2 v2 - Joint Training Results\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Architecture: ResNet-18 (random init)\n")
    f.write(f"Train samples: {len(train_dataset):,}\n")
    f.write(f"Val samples: {len(val_dataset):,}\n")
    f.write(f"Epochs trained: {len(history['train_acc'])}\n")
    f.write(f"Best Val Accuracy: {best_val_acc:.2f}%\n")
    f.write(f"Final Train Accuracy: {history['train_acc'][-1]:.2f}%\n")
    f.write(f"Final Val Accuracy: {history['val_acc'][-1]:.2f}%\n")
    f.write(f"\nHyperparameters:\n")
    f.write(f"  Mask LR: {MASK_LR}\n")
    f.write(f"  Classifier LR: {CLASSIFIER_LR}\n")
    f.write(f"  Mask regularization: {MASK_REG_WEIGHT}\n")
    f.write(f"  Data augmentation: Yes\n")

print("Results saved!")
print(f"  {RESULTS_DIR}")

## 10. Summary

In [ ]:
print("=" * 70)
print("RESNET-18 PHASE 2 v2 SUMMARY")
print("=" * 70)
print(f"\nTraining Setup:")
print(f"  Train/Val Split: {len(train_dataset):,} / {len(val_dataset):,}")
print(f"  Data Augmentation: RandomResizedCrop, HorizontalFlip, ColorJitter")
print(f"  Mask Regularization: {MASK_REG_WEIGHT}")
print(f"\nResults:")
print(f"  Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"  Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"  Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
print(f"\nArtifacts: {RESULTS_DIR}")
print("\nNext: Compare mask with Phase 1 to determine if frequency preference is architectural")